# Camada Gold - modelagem dimensional e métricas de negócio

## Modelo estrela

```
                     dim_municipio
                           |
   dim_empresa  ---  fato_acessos  ---  dim_tecnologia
                      /         \
         dim_faixa_velocidade    dim_segmento
```

**Grão da fato:** município × empresa × tecnologia+meio × faixa de velocidade ×
segmento, para o período de referência.

## Decisões de modelagem

**Chaves surrogate determinísticas.** Geradas por `row_number()` sobre a ordenação
da chave natural. Reexecutar o notebook produz as mesmas chaves.

**`dim_tecnologia` tem chave composta** (`tecnologia` + `meio_acesso`). A tecnologia
não determina funcionalmente o meio físico — ETHERNET aparece sobre fibra e sobre
cabo metálico. Um join apenas por `tecnologia` duplicaria linhas e inflaria os
acessos.

**Dimensão lixo (junk dimension).** `tipo_pessoa` e `tipo_produto` são combinados em
`dim_segmento`, padrão de Kimball para atributos de baixa cardinalidade. Ela carrega
as flags que recortam os numeradores.

**Sem dimensão de tempo.** Período único; uma dimensão de uma
linha seria degenerada. O período é atributo da fato.

**Sem SCD Tipo 2.** Sem histórico, não há mudança a versionar.

## Todos os acessos, dois denominadores

Nenhum tipo de acesso é descartado. A tabela analítica expõe sete numeradores —
total, pessoa física, pessoa jurídica, residencial estrito, INTERNET, linha dedicada
e M2M — porque toda conexão ativa é mercado atendido.

E dois denominadores, porque a escolha muda a leitura:

| Denominador | Origem | Leitura |
|---|---|---|
| `domicilios_ocupados_qtd` | SIDRA 4712 v/381 | conservador, comparável ao padrão IBGE |
| `domicilios_total_qtd` | SIDRA 4711 c3/59993 | mercado endereçável real, inclui uso ocasional |

A diferença entre as duas penetrações mede o efeito dos imóveis de veraneio.

## Convenção de nomes

`_qtd` contagem · `_hab` habitantes · `_pct` percentual · `_brl` reais ·
`_mil_brl` mil reais · `_km2` quilômetros quadrados

## 1. Parâmetros e funções

In [0]:
CATALOGO_SILVER = "silver"
CATALOGO_GOLD   = "gold"
SCHEMA          = "telecom"

LIMITE_DEFASAGEM_MBPS = 34      # abaixo disso, faixa considerada defasada
HHI_ALTA              = 2500    # limiares usuais de defesa da concorrência
HHI_MODERADA          = 1500
PENETRACAO_SATURADA   = 90

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO_GOLD}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO_GOLD}.{SCHEMA}")
print(f"Destino: {CATALOGO_GOLD}.{SCHEMA}")

In [0]:
def br(coluna, decimais: int = 0, sufixo: str = ""):
    """Formata número no padrão brasileiro: 1.474 / 26,46 / 26,46 %"""
    txt = F.format_number(F.col(coluna), decimais)
    txt = F.translate(txt, ".,", ",.")
    return F.concat(txt, F.lit(sufixo)) if sufixo else txt


def exibir(df, numericos: dict, texto: list = None):
    """numericos: {"coluna": (casas_decimais, sufixo)}"""
    texto = texto or []
    sel  = [F.col(x) for x in texto]
    sel += [br(col, d, s).alias(col) for col, (d, s) in numericos.items()]
    return df.select(*sel)


def gravar_gold(df, tabela: str, comentario: str):
    df = df.withColumn("_data_processamento",
                       F.lit(datetime.now(timezone.utc).isoformat()))
    nome = f"{CATALOGO_GOLD}.{SCHEMA}.{tabela}"
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(nome))
    spark.sql(f"COMMENT ON TABLE {nome} IS '{comentario.replace(chr(39), '')}'")
    print(f"{nome}: {spark.table(nome).count():,} linhas / {len(df.columns)} colunas")
    return spark.table(nome)


def documentar_colunas(tabela: str, dicionario: dict):
    nome = f"{CATALOGO_GOLD}.{SCHEMA}.{tabela}"
    for coluna, descricao in dicionario.items():
        spark.sql(f"COMMENT ON COLUMN {nome}.{coluna} IS "
                  f"'{descricao.replace(chr(39), '')}'")
    print(f"{nome}: {len(dicionario)} colunas documentadas")


def chave_surrogate(df, ordem: list, nome_chave: str):
    """Chave surrogate determinística a partir da ordenação da chave natural."""
    return df.withColumn(nome_chave,
        F.row_number().over(Window.orderBy(*[F.col(x) for x in ordem])))

In [0]:
s_acessos     = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.acessos")
s_empresas    = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.empresas")
s_tecnologias = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.tecnologias")
s_municipios  = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.municipios")

print(f"acessos={s_acessos.count():,} | empresas={s_empresas.count()} | "
      f"tecnologias={s_tecnologias.count()} | municipios={s_municipios.count()}")

## 2. Dimensões

### 2.1 dim_municipio

Sustenta análises em quatro níveis geográficos e
carrega os dois denominadores de mercado, mais os atributos socioeconômicos.

In [0]:
dim_municipio = chave_surrogate(
    s_municipios.select(
        "codigo_ibge", "nome_municipio",
        "microrregiao_nome", "mesorregiao_nome",
        "regiao_imediata_nome", "regiao_intermediaria_nome", "uf_sigla",
        "porte_populacional",
        "populacao_residente_hab", "area_km2", "densidade_demografica_hab_km2",
        "domicilios_total_qtd", "domicilios_ocupados_qtd",
        "domicilios_nao_ocupados_qtd", "domicilios_vagos_qtd",
        "domicilios_uso_ocasional_qtd", "uso_ocasional_pct", "perfil_ocupacao",
        "moradores_em_domicilios_qtd", "media_moradores_domicilio",
        "pib_total_mil_brl", "pib_per_capita_brl", "vab_total_mil_brl",
        "vab_agropecuaria_mil_brl", "vab_industria_mil_brl",
        "vab_servicos_mil_brl", "vab_administracao_mil_brl",
        "setor_dominante",
    ),
    ordem=["codigo_ibge"], nome_chave="sk_municipio"
)

gravar_gold(dim_municipio, "dim_municipio",
    "Dimensao de municipios do RS. Hierarquia geografica do IBGE e atributos "
    "socioeconomicos do Censo 2022 e do PIB Municipal. Traz DOIS denominadores de "
    "mercado: domicilios_ocupados_qtd (conservador) e domicilios_total_qtd (mercado "
    "enderecavel, inclui imoveis de uso ocasional). Setor dominante definido pelo "
    "maior VAB setorial. Chave surrogate deterministica sk_municipio.")

In [0]:
documentar_colunas("dim_municipio", {
    "sk_municipio": "Chave surrogate. Inteiro sequencial deterministico gerado pela "
        "ordenacao do codigo IBGE.",
    "codigo_ibge": "Codigo IBGE do municipio, 7 digitos. Chave natural. "
        "Dominio: 4300034 a 4323804. Linhagem: IBGE API Localidades.",
    "nome_municipio": "Nome oficial do municipio.",
    "microrregiao_nome": "Microrregiao geografica do IBGE.",
    "mesorregiao_nome": "Mesorregiao geografica. Dominio: 7 categorias no RS.",
    "regiao_imediata_nome": "Regiao geografica imediata, divisao vigente.",
    "regiao_intermediaria_nome": "Regiao geografica intermediaria do IBGE.",
    "uf_sigla": "Sigla da unidade federativa. Valor unico: RS.",
    "porte_populacional": "Faixa de porte. Dominio: Ate 5 mil, 5 a 20 mil, "
        "20 a 100 mil, Acima de 100 mil.",
    "populacao_residente_hab": "Populacao residente. Unidade: habitantes. "
        "Linhagem: SIDRA 4714 v/93, Censo 2022.",
    "area_km2": "Area territorial. Unidade: quilometros quadrados. "
        "Linhagem: SIDRA 4714 v/6318.",
    "densidade_demografica_hab_km2": "Densidade demografica. Unidade: habitantes por "
        "quilometro quadrado. Linhagem: SIDRA 4714 v/614.",
    "domicilios_total_qtd": "Total de domicilios recenseados, todas as especies. "
        "Unidade: domicilios. Denominador de mercado enderecavel: qualquer imovel e "
        "potencial contratante. Linhagem: SIDRA 4711 v/617 c3/59993, Censo 2022.",
    "domicilios_ocupados_qtd": "Domicilios particulares permanentes ocupados. "
        "Unidade: domicilios. Denominador conservador. Linhagem: SIDRA 4712 v/381.",
    "domicilios_nao_ocupados_qtd": "Domicilios permanentes nao ocupados, soma de "
        "vagos e uso ocasional. Unidade: domicilios. Linhagem: SIDRA 4711 c3/60001.",
    "domicilios_vagos_qtd": "Domicilios nao ocupados classificados como vagos. "
        "Unidade: domicilios. Baixa probabilidade de contrato ativo. "
        "Linhagem: SIDRA 4711 c3/60002.",
    "domicilios_uso_ocasional_qtd": "Domicilios nao ocupados de uso ocasional, "
        "tipicos de veraneio e turismo. Unidade: domicilios. Alta probabilidade de "
        "contrato ativo mesmo sem morador permanente. Linhagem: SIDRA 4711 c3/60003.",
    "uso_ocasional_pct": "Participacao de imoveis de uso ocasional no total "
        "recenseado. Unidade: percentual. Dominio: 0 a 100.",
    "perfil_ocupacao": "Classificacao pelo peso do uso ocasional. Dominio: Ocupacao "
        "permanente (abaixo de 10 por cento), Ocupacao mista (10 a 29), Veraneio ou "
        "turismo (30 ou mais).",
    "moradores_em_domicilios_qtd": "Moradores em domicilios particulares permanentes "
        "ocupados. Unidade: pessoas. Linhagem: SIDRA 4712 v/382.",
    "media_moradores_domicilio": "Media de moradores por domicilio. Unidade: "
        "moradores por domicilio. Linhagem: SIDRA 4712 v/5930.",
    "pib_total_mil_brl": "PIB a precos correntes. Unidade: mil reais. "
        "Linhagem: SIDRA 5938 v/37.",
    "pib_per_capita_brl": "PIB por habitante. Unidade: reais.",
    "vab_total_mil_brl": "Valor adicionado bruto total. Unidade: mil reais.",
    "vab_agropecuaria_mil_brl": "VAB da agropecuaria. Unidade: mil reais.",
    "vab_industria_mil_brl": "VAB da industria. Unidade: mil reais.",
    "vab_servicos_mil_brl": "VAB dos servicos, exclusive administracao publica. "
        "Unidade: mil reais.",
    "vab_administracao_mil_brl": "VAB da administracao, defesa, educacao e saude "
        "publicas. Unidade: mil reais.",
    "setor_dominante": "Setor com maior VAB. Dominio: Agropecuaria, Industria, "
        "Servicos, Administracao publica.",
})

In [0]:
display(exibir(
    dim_municipio.orderBy(F.col("populacao_residente_hab").desc()).limit(15),
    numericos={"sk_municipio": (0, ""),
               "populacao_residente_hab": (0, ""),
               "domicilios_total_qtd": (0, ""),
               "domicilios_ocupados_qtd": (0, ""),
               "domicilios_uso_ocasional_qtd": (0, ""),
               "uso_ocasional_pct": (2, " %"),
               "pib_per_capita_brl": (2, "")},
    texto=["codigo_ibge", "nome_municipio", "mesorregiao_nome",
           "porte_populacional", "perfil_ocupacao", "setor_dominante"]))

### 2.2 dim_empresa

Chave natural: CNPJ. O grupo econômico é atributo, não chave de análise — a Anatel
classifica todos os provedores regionais como `OUTROS`, 55% das linhas do RS.

In [0]:
dim_empresa = chave_surrogate(
    s_empresas.select("cnpj", "empresa", "grupo_economico", "porte",
                      "grupo_identificado"),
    ordem=["cnpj"], nome_chave="sk_empresa"
)

gravar_gold(dim_empresa, "dim_empresa",
    "Dimensao de prestadoras de SCM atuantes no RS. Chave natural: CNPJ. O campo "
    "grupo_economico traz OUTROS para todos os provedores regionais, o que o "
    "inviabiliza como chave de analise competitiva.")

documentar_colunas("dim_empresa", {
    "sk_empresa": "Chave surrogate deterministica gerada pela ordenacao do CNPJ.",
    "cnpj": "CNPJ da prestadora, 14 digitos sem formatacao. Chave natural e chave de "
        "identidade competitiva.",
    "empresa": "Nome comercial da prestadora. Determinado funcionalmente pelo CNPJ.",
    "grupo_economico": "Grupo economico declarado. Dominio: 16 categorias no RS. "
        "OUTROS agrega todos os provedores regionais, 55 por cento das linhas.",
    "porte": "Porte segundo a Anatel. Dominio: Pequeno Porte, Grande Porte.",
    "grupo_identificado": "Indica grupo diferente de OUTROS. Dominio: true, false.",
})

### 2.3 dim_tecnologia

**Chave composta:** `tecnologia` + `meio_acesso`. A tecnologia não determina o meio
físico; usar apenas `tecnologia` produziria chave duplicada e inflaria os acessos.

In [0]:
dim_tecnologia = chave_surrogate(
    s_tecnologias.select("tecnologia", "meio_acesso", "e_fibra"),
    ordem=["tecnologia", "meio_acesso"], nome_chave="sk_tecnologia"
)

gravar_gold(dim_tecnologia, "dim_tecnologia",
    "Dimensao de tecnologias de acesso. Chave composta: tecnologia + meio_acesso. A "
    "tecnologia NAO determina funcionalmente o meio fisico: ETHERNET aparece sobre "
    "fibra e sobre cabo metalico. A flag e_fibra sustenta as analises de fibra.")

documentar_colunas("dim_tecnologia", {
    "sk_tecnologia": "Chave surrogate deterministica gerada pela ordenacao de "
        "tecnologia e meio de acesso.",
    "tecnologia": "Tecnologia de acesso declarada. Parte da chave natural composta. "
        "Dominio: 24 categorias, entre elas FTTH, ETHERNET, HFC, VSAT, ADSL2.",
    "meio_acesso": "Meio fisico de transmissao. Parte da chave natural composta. "
        "Dominio: Fibra, Radio, Satelite, Cabo Metalico, Cabo Coaxial.",
    "e_fibra": "Indica meio de acesso igual a Fibra. Dominio: true, false.",
})

print(f"Combinacoes: {dim_tecnologia.count()} | "
      f"Tecnologias distintas: {dim_tecnologia.select('tecnologia').distinct().count()}")
display(dim_tecnologia.orderBy("meio_acesso", "tecnologia"))

### 2.4 dim_faixa_velocidade

Dimensão com **ordem explícita**. As faixas são categorias de texto sem ordenação
natural — sem `ordem_faixa`, qualquer gráfico sai alfabético, com `> 34Mbps` antes de
`0Kbps a 512Kbps`.

In [0]:
faixas = [
    ("0Kbps a 512Kbps",  1, True),
    ("512kbps a 2Mbps",  2, True),
    ("2Mbps a 12Mbps",   3, True),
    ("12Mbps a 34Mbps",  4, True),
    ("> 34Mbps",         5, False),
]

dim_faixa = (
    spark.createDataFrame(faixas, ["faixa_velocidade", "ordem_faixa", "e_defasada"])
    .withColumn("sk_faixa", F.col("ordem_faixa"))
)

gravar_gold(dim_faixa, "dim_faixa_velocidade",
    "Dimensao de faixas de velocidade contratada. A coluna ordem_faixa estabelece a "
    "ordenacao correta das categorias, que nao e alfabetica. A flag e_defasada marca "
    "faixas abaixo de 34 Mbps e sustenta o calculo de defasagem tecnologica.")

documentar_colunas("dim_faixa_velocidade", {
    "sk_faixa": "Chave surrogate, igual a ordem_faixa. Dominio: 1 a 5.",
    "faixa_velocidade": "Faixa de velocidade contratada. Chave natural. Dominio: as "
        "5 categorias declaradas pela Anatel.",
    "ordem_faixa": "Ordem crescente de velocidade. Unidade: posicao ordinal. "
        "Dominio: 1 (mais lenta) a 5 (mais rapida).",
    "e_defasada": "Indica faixa abaixo de 34 Mbps. Dominio: true, false.",
})

display(dim_faixa.orderBy("ordem_faixa"))

### 2.5 dim_segmento (dimensão lixo)

Combina `tipo_pessoa` e `tipo_produto`. As flags recortam os sete numeradores da
tabela analítica sem que nenhum acesso seja descartado.

In [0]:
dim_segmento = chave_surrogate(
    s_acessos.select("tipo_pessoa", "tipo_produto").distinct()
    # rlike tolera "Pessoa Fisica" e "Pessoa Física"
    .withColumn("e_pessoa_fisica",   F.col("tipo_pessoa").rlike("^Pessoa F"))
    .withColumn("e_pessoa_juridica", ~F.col("tipo_pessoa").rlike("^Pessoa F"))
    .withColumn("e_internet",        F.col("tipo_produto") == "INTERNET")
    .withColumn("e_linha_dedicada",  F.col("tipo_produto") == "LINHA_DEDICADA")
    .withColumn("e_m2m",             F.col("tipo_produto") == "M2M")
    .withColumn("e_residencial",
        F.col("tipo_pessoa").rlike("^Pessoa F") & (F.col("tipo_produto") == "INTERNET")),
    ordem=["tipo_pessoa", "tipo_produto"], nome_chave="sk_segmento"
)

gravar_gold(dim_segmento, "dim_segmento",
    "Dimensao lixo combinando tipo de pessoa e tipo de produto, ambos de baixa "
    "cardinalidade. As flags recortam os numeradores da tabela analitica sem "
    "descartar nenhum acesso: pessoa fisica, pessoa juridica, INTERNET, linha "
    "dedicada, M2M e residencial estrito (pessoa fisica com produto INTERNET).")

documentar_colunas("dim_segmento", {
    "sk_segmento": "Chave surrogate deterministica.",
    "tipo_pessoa": "Natureza do assinante. Dominio: Pessoa Fisica, Pessoa Juridica.",
    "tipo_produto": "Tipo de produto. Dominio: INTERNET, LINHA_DEDICADA, M2M, OUTROS.",
    "e_pessoa_fisica": "Indica assinante pessoa fisica, qualquer produto. "
        "Dominio: true, false.",
    "e_pessoa_juridica": "Indica assinante pessoa juridica, qualquer produto. "
        "Dominio: true, false.",
    "e_internet": "Indica produto INTERNET, qualquer tipo de pessoa. "
        "Dominio: true, false.",
    "e_linha_dedicada": "Indica produto LINHA_DEDICADA. Dominio: true, false.",
    "e_m2m": "Indica produto M2M, comunicacao entre maquinas. Dominio: true, false.",
    "e_residencial": "Indica pessoa fisica com produto INTERNET, recorte residencial "
        "estrito. Dominio: true, false.",
})

display(dim_segmento.orderBy("tipo_pessoa", "tipo_produto"))

## 3. Tabela de fatos

Substitui as chaves naturais pelas surrogate. A medida `acessos_qtd` é aditiva.

Note o join de `dim_tecnologia` pelas **duas** colunas da chave composta.

In [0]:
fato_acessos = (
    s_acessos
    .join(dim_municipio.select("sk_municipio", "codigo_ibge"), "codigo_ibge")
    .join(dim_empresa.select("sk_empresa", "cnpj"), "cnpj")
    .join(dim_tecnologia.select("sk_tecnologia", "tecnologia", "meio_acesso"),
          ["tecnologia", "meio_acesso"])
    .join(dim_faixa.select("sk_faixa", "faixa_velocidade"), "faixa_velocidade")
    .join(dim_segmento.select("sk_segmento", "tipo_pessoa", "tipo_produto"),
          ["tipo_pessoa", "tipo_produto"])
    .select("periodo", "sk_municipio", "sk_empresa", "sk_tecnologia",
            "sk_faixa", "sk_segmento", "acessos_qtd")
)

gravar_gold(fato_acessos, "fato_acessos",
    "Tabela de fatos de acessos de banda larga fixa no RS. Grao: municipio x empresa "
    "x tecnologia com meio de acesso x faixa de velocidade x segmento, para um "
    "periodo de referencia. Medida aditiva: acessos_qtd. O periodo e atributo "
    "degenerado da fato, ja que o MVP trabalha com snapshot unico.")

documentar_colunas("fato_acessos", {
    "periodo": "Periodo de referencia no formato AAAA-MM. Atributo degenerado. "
        "Valor unico neste MVP: 2026-07.",
    "sk_municipio": "Chave estrangeira para gold.dim_municipio.",
    "sk_empresa": "Chave estrangeira para gold.dim_empresa.",
    "sk_tecnologia": "Chave estrangeira para gold.dim_tecnologia, cuja chave natural "
        "na origem e composta por tecnologia e meio de acesso.",
    "sk_faixa": "Chave estrangeira para gold.dim_faixa_velocidade.",
    "sk_segmento": "Chave estrangeira para gold.dim_segmento.",
    "acessos_qtd": "Quantidade de acessos ativos. Unidade: acessos. Medida aditiva "
        "em todas as dimensoes. Linhagem: silver.acessos.acessos_qtd.",
})

### Validação de integridade referencial

Nenhuma linha e nenhum acesso pode se perder ou se multiplicar nos cinco joins.
Divergência aqui indica chave duplicada em alguma dimensão.

In [0]:
linhas_origem  = s_acessos.count()
linhas_destino = fato_acessos.count()
soma_origem    = s_acessos.agg(F.sum("acessos_qtd")).first()[0]
soma_destino   = fato_acessos.agg(F.sum("acessos_qtd")).first()[0]

print(f"{'':<10} {'SILVER':>14} {'GOLD':>14}  INTEGRO")
print("-" * 52)
print(f"{'Linhas':<10} {linhas_origem:>14,} {linhas_destino:>14,}  {linhas_origem == linhas_destino}")
print(f"{'Acessos':<10} {soma_origem:>14,} {soma_destino:>14,}  {soma_origem == soma_destino}")

print("\nChaves estrangeiras nulas:")
for sk in ["sk_municipio", "sk_empresa", "sk_tecnologia", "sk_faixa", "sk_segmento"]:
    print(f"  {sk:<15} {fato_acessos.filter(F.col(sk).isNull()).count()}")

## 4. Tabela analítica por município

Uma linha por município, com sete numeradores e dois denominadores.

### Numeradores

| Coluna | Recorte |
|---|---|
| `acessos_total_qtd` | todos os acessos |
| `acessos_pf_qtd` | pessoa física, qualquer produto |
| `acessos_pj_qtd` | pessoa jurídica, qualquer produto |
| `acessos_residencial_qtd` | pessoa física + INTERNET |
| `acessos_internet_qtd` | produto INTERNET, PF e PJ |
| `acessos_dedicado_qtd` | linha dedicada |
| `acessos_m2m_qtd` | comunicação entre máquinas |

### Penetrações

| Métrica | Fórmula |
|---|---|
| `penetracao_ocupados_pct` | acessos totais ÷ domicílios ocupados × 100 |
| `penetracao_enderecavel_pct` | acessos totais ÷ domicílios recenseados × 100 |
| `penetracao_residencial_pct` | acessos residenciais ÷ domicílios recenseados × 100 |
| `contratos_por_domicilio` | acessos totais ÷ domicílios recenseados |

`contratos_por_domicilio` é a mesma razão da penetração endereçável, em escala
unitária. A leitura muda: "1,32 contratos por domicílio" descreve intensidade
competitiva, enquanto "132% de penetração" parece anomalia.

### Estrutura competitiva

`n_provedores_qtd`, `share_lider_pct`, `hhi` e sua classificação. O HHI é a medida
padrão usada por autoridades de defesa da concorrência: acima de 2.500 indica
mercado altamente concentrado; abaixo de 1.500, desconcentrado.

In [0]:
f = fato_acessos

base = (
    f.join(dim_segmento.select("sk_segmento", "e_pessoa_fisica", "e_pessoa_juridica",
                               "e_internet", "e_linha_dedicada", "e_m2m",
                               "e_residencial"), "sk_segmento")
     .join(dim_tecnologia.select("sk_tecnologia", "e_fibra"), "sk_tecnologia")
     .join(dim_faixa.select("sk_faixa", "e_defasada"), "sk_faixa")
)

def soma_se(flag, alias):
    return F.sum(F.when(F.col(flag), F.col("acessos_qtd")).otherwise(0)).alias(alias)

agregado = base.groupBy("sk_municipio").agg(
    F.sum("acessos_qtd").alias("acessos_total_qtd"),
    soma_se("e_pessoa_fisica",   "acessos_pf_qtd"),
    soma_se("e_pessoa_juridica", "acessos_pj_qtd"),
    soma_se("e_residencial",     "acessos_residencial_qtd"),
    soma_se("e_internet",        "acessos_internet_qtd"),
    soma_se("e_linha_dedicada",  "acessos_dedicado_qtd"),
    soma_se("e_m2m",             "acessos_m2m_qtd"),
    soma_se("e_fibra",           "acessos_fibra_qtd"),
    soma_se("e_defasada",        "acessos_defasados_qtd"),
)

In [0]:
# Estrutura competitiva: share de cada CNPJ dentro do municipio
por_municipio = Window.partitionBy("sk_municipio")

share_empresa = (
    f.groupBy("sk_municipio", "sk_empresa")
     .agg(F.sum("acessos_qtd").alias("acessos_empresa_qtd"))
     .withColumn("total_municipio_qtd", F.sum("acessos_empresa_qtd").over(por_municipio))
     .withColumn("share_pct",
         100 * F.col("acessos_empresa_qtd") / F.col("total_municipio_qtd"))
     .withColumn("posicao",
         F.row_number().over(por_municipio.orderBy(F.col("acessos_empresa_qtd").desc())))
)

competitivo = share_empresa.groupBy("sk_municipio").agg(
    F.countDistinct("sk_empresa").alias("n_provedores_qtd"),
    F.round(F.sum(F.pow(F.col("share_pct"), 2)), 1).alias("hhi"),
    F.round(F.max("share_pct"), 2).alias("share_lider_pct"),
    F.first(F.when(F.col("posicao") == 1, F.col("sk_empresa")),
            ignorenulls=True).alias("sk_lider"),
)

# Faixa de velocidade com maior volume de acessos no municipio
faixa_modal = (
    f.groupBy("sk_municipio", "sk_faixa")
     .agg(F.sum("acessos_qtd").alias("acessos_faixa_qtd"))
     .withColumn("rn", F.row_number().over(
         Window.partitionBy("sk_municipio").orderBy(F.col("acessos_faixa_qtd").desc())))
     .filter(F.col("rn") == 1)
     .join(dim_faixa.select("sk_faixa", "faixa_velocidade"), "sk_faixa")
     .select("sk_municipio", F.col("faixa_velocidade").alias("faixa_modal"))
)

In [0]:
gold_mercado = (
    dim_municipio
    .join(agregado,    "sk_municipio", "left")
    .join(competitivo, "sk_municipio", "left")
    .join(faixa_modal, "sk_municipio", "left")
    .join(dim_empresa.select(F.col("sk_empresa").alias("sk_lider"),
                             F.col("empresa").alias("provedor_lider")),
          "sk_lider", "left")
    # --- penetracoes ---
    .withColumn("penetracao_ocupados_pct",
        F.round(100 * F.col("acessos_total_qtd") / F.col("domicilios_ocupados_qtd"), 2))
    .withColumn("penetracao_enderecavel_pct",
        F.round(100 * F.col("acessos_total_qtd") / F.col("domicilios_total_qtd"), 2))
    .withColumn("penetracao_residencial_pct",
        F.round(100 * F.col("acessos_residencial_qtd") / F.col("domicilios_total_qtd"), 2))
    # Mesma razao da penetracao enderecavel, lida como intensidade competitiva:
    # 1,32 contratos por domicilio e mais informativo que 132 por cento de penetracao.
    .withColumn("contratos_por_domicilio",
        F.round(F.col("acessos_total_qtd") / F.col("domicilios_total_qtd"), 2))
    # --- mercado nao atendido, sobre o denominador enderecavel ---
    .withColumn("saldo_domicilios_qtd",
        F.col("domicilios_total_qtd") - F.col("acessos_total_qtd"))
    .withColumn("domicilios_sem_acesso_qtd",
        F.greatest(F.lit(0), F.col("saldo_domicilios_qtd")))
    .withColumn("penetracao_acima_100",
        F.col("penetracao_enderecavel_pct") > 100)
    .withColumn("mercado_saturado",
        F.col("penetracao_enderecavel_pct") >= PENETRACAO_SATURADA)
    # --- perfil tecnologico ---
    .withColumn("fibra_pct",
        F.round(100 * F.col("acessos_fibra_qtd") / F.col("acessos_total_qtd"), 2))
    .withColumn("defasagem_pct",
        F.round(100 * F.col("acessos_defasados_qtd") / F.col("acessos_total_qtd"), 2))
    # --- concentracao ---
    .withColumn("concentracao",
        F.when(F.col("hhi") >= HHI_ALTA,     "Alta")
         .when(F.col("hhi") >= HHI_MODERADA, "Moderada")
         .otherwise("Baixa"))
    .select(
        "sk_municipio", "codigo_ibge", "nome_municipio",
        "mesorregiao_nome", "regiao_intermediaria_nome", "porte_populacional",
        "populacao_residente_hab", "area_km2", "densidade_demografica_hab_km2",
        "domicilios_total_qtd", "domicilios_ocupados_qtd",
        "domicilios_uso_ocasional_qtd", "uso_ocasional_pct", "perfil_ocupacao",
        "pib_per_capita_brl", "setor_dominante",
        "acessos_total_qtd", "acessos_pf_qtd", "acessos_pj_qtd",
        "acessos_residencial_qtd", "acessos_internet_qtd",
        "acessos_dedicado_qtd", "acessos_m2m_qtd",
        "penetracao_ocupados_pct", "penetracao_enderecavel_pct",
        "penetracao_residencial_pct", "contratos_por_domicilio",
        "saldo_domicilios_qtd", "domicilios_sem_acesso_qtd",
        "penetracao_acima_100", "mercado_saturado",
        "n_provedores_qtd", "provedor_lider", "share_lider_pct", "hhi", "concentracao",
        "fibra_pct", "defasagem_pct", "faixa_modal",
    )
)

gravar_gold(gold_mercado, "mercado_municipio",
    "Tabela analitica com uma linha por municipio do RS. Expoe sete numeradores de "
    "acesso (total, pessoa fisica, pessoa juridica, residencial estrito, INTERNET, "
    "linha dedicada e M2M) e dois denominadores de mercado. A penetracao enderecavel "
    "usa domicilios recenseados como denominador, o que corrige a distorcao em "
    "municipios de veraneio, onde imoveis de uso ocasional tem contrato ativo mas "
    "ficam fora da contagem de domicilios ocupados do Censo. HHI = soma dos quadrados "
    "dos shares por CNPJ, escala 0 a 10000.")

In [0]:
documentar_colunas("mercado_municipio", {
    "sk_municipio": "Chave surrogate do municipio. Linhagem: gold.dim_municipio.",
    "codigo_ibge": "Codigo IBGE do municipio, 7 digitos. Chave natural.",
    "nome_municipio": "Nome oficial do municipio.",
    "mesorregiao_nome": "Mesorregiao geografica do IBGE. Dominio: 7 categorias no RS.",
    "regiao_intermediaria_nome": "Regiao geografica intermediaria do IBGE.",
    "porte_populacional": "Faixa de porte populacional. Dominio: Ate 5 mil, "
        "5 a 20 mil, 20 a 100 mil, Acima de 100 mil.",
    "populacao_residente_hab": "Populacao residente. Unidade: habitantes. "
        "Linhagem: SIDRA 4714 v/93, Censo 2022.",
    "area_km2": "Area da unidade territorial. Unidade: quilometros quadrados. "
        "Linhagem: SIDRA 4714 v/6318.",
    "densidade_demografica_hab_km2": "Densidade demografica. Unidade: habitantes por "
        "quilometro quadrado. Proxy de custo de rede por assinante: quanto menor a "
        "densidade, maior o custo de cobertura. Linhagem: SIDRA 4714 v/614.",
    "domicilios_total_qtd": "Total de domicilios recenseados, todas as especies. "
        "Unidade: domicilios. Denominador de mercado enderecavel. "
        "Linhagem: SIDRA 4711 v/617 c3/59993.",
    "domicilios_ocupados_qtd": "Domicilios particulares permanentes ocupados. "
        "Unidade: domicilios. Denominador conservador. Linhagem: SIDRA 4712 v/381.",
    "domicilios_uso_ocasional_qtd": "Domicilios de uso ocasional, tipicos de veraneio. "
        "Unidade: domicilios. Linhagem: SIDRA 4711 c3/60003.",
    "uso_ocasional_pct": "Participacao de imoveis de uso ocasional no total. "
        "Unidade: percentual. Dominio: 0 a 100.",
    "perfil_ocupacao": "Classificacao pelo peso do uso ocasional. Dominio: Ocupacao "
        "permanente, Ocupacao mista, Veraneio ou turismo.",
    "pib_per_capita_brl": "PIB por habitante. Unidade: reais. Linhagem: SIDRA 5938.",
    "setor_dominante": "Setor com maior VAB. Dominio: Agropecuaria, Industria, "
        "Servicos, Administracao publica.",
    "acessos_total_qtd": "Total de acessos de banda larga fixa, todos os tipos. "
        "Unidade: acessos. Linhagem: soma de gold.fato_acessos por municipio.",
    "acessos_pf_qtd": "Acessos de pessoa fisica, qualquer produto. Unidade: acessos. "
        "Linhagem: fato_acessos filtrado por dim_segmento.e_pessoa_fisica.",
    "acessos_pj_qtd": "Acessos de pessoa juridica, qualquer produto. "
        "Unidade: acessos. Linhagem: filtro por dim_segmento.e_pessoa_juridica.",
    "acessos_residencial_qtd": "Acessos de pessoa fisica com produto INTERNET, "
        "recorte residencial estrito. Unidade: acessos.",
    "acessos_internet_qtd": "Acessos com produto INTERNET, pessoa fisica e juridica. "
        "Unidade: acessos.",
    "acessos_dedicado_qtd": "Acessos com produto LINHA_DEDICADA. Unidade: acessos. "
        "Tipicamente corporativos de maior valor.",
    "acessos_m2m_qtd": "Acessos com produto M2M, comunicacao entre maquinas. "
        "Unidade: acessos.",
    "penetracao_ocupados_pct": "Acessos totais sobre domicilios ocupados. Unidade: "
        "percentual. Denominador conservador; superestima municipios de veraneio.",
    "penetracao_enderecavel_pct": "Acessos totais sobre domicilios recenseados. "
        "Unidade: percentual. Metrica principal de penetracao deste MVP.",
    "penetracao_residencial_pct": "Acessos residenciais estritos sobre domicilios "
        "recenseados. Unidade: percentual.",
    "contratos_por_domicilio": "Acessos totais divididos por domicilios recenseados. "
        "Unidade: contratos por domicilio. Mesma razao de penetracao_enderecavel_pct, "
        "em escala unitaria. Valores acima de 1 indicam multiplos contratos por "
        "imovel, tipicos de mercados saturados com competicao intensa.",
    "saldo_domicilios_qtd": "Diferenca entre domicilios recenseados e acessos totais. "
        "Unidade: domicilios. Pode ser negativo.",
    "domicilios_sem_acesso_qtd": "Mercado nao atendido, com piso em zero. "
        "Unidade: domicilios. Linhagem: maior valor entre 0 e saldo_domicilios_qtd.",
    "penetracao_acima_100": "Sinaliza penetracao enderecavel acima de 100 por cento. "
        "Dominio: true, false. Indica multiplos contratos por imovel ou acessos "
        "declarados no municipio da prestadora.",
    "mercado_saturado": "Sinaliza penetracao enderecavel igual ou superior a 90 por "
        "cento. Dominio: true, false.",
    "n_provedores_qtd": "CNPJs distintos com acesso declarado. Unidade: prestadoras. "
        "Valor 1 indica monopolio de fato.",
    "provedor_lider": "Nome da prestadora com maior numero de acessos no municipio.",
    "share_lider_pct": "Participacao do provedor lider. Unidade: percentual. "
        "Dominio: 0 a 100.",
    "hhi": "Indice Herfindahl-Hirschman. Unidade: indice adimensional. Dominio: 0 a "
        "10000. Formula: soma dos quadrados dos shares percentuais por CNPJ.",
    "concentracao": "Classificacao do HHI. Dominio: Baixa (abaixo de 1500), Moderada "
        "(1500 a 2499), Alta (2500 ou mais). Limiares de defesa da concorrencia.",
    "fibra_pct": "Participacao da fibra optica nos acessos. Unidade: percentual. "
        "Dominio: 0 a 100.",
    "defasagem_pct": "Participacao de acessos abaixo de 34 Mbps. Unidade: percentual. "
        "Dominio: 0 a 100.",
    "faixa_modal": "Faixa de velocidade com maior volume de acessos no municipio.",
})

Exibição formatada

In [0]:
display(exibir(
    gold_mercado.orderBy(F.col("populacao_residente_hab").desc()).limit(25),
    numericos={"populacao_residente_hab":     (0, ""),
               "domicilios_total_qtd":        (0, ""),
               "domicilios_ocupados_qtd":     (0, ""),
               "acessos_total_qtd":           (0, ""),
               "acessos_pf_qtd":              (0, ""),
               "acessos_pj_qtd":              (0, ""),
               "penetracao_ocupados_pct":     (2, " %"),
               "penetracao_enderecavel_pct":  (2, " %"),
               "contratos_por_domicilio":     (2, ""),
               "domicilios_sem_acesso_qtd":   (0, ""),
               "share_lider_pct":             (2, " %"),
               "hhi":                         (1, ""),
               "fibra_pct":                   (2, " %"),
               "defasagem_pct":               (2, " %"),
               "pib_per_capita_brl":          (2, "")},
    texto=["nome_municipio", "mesorregiao_nome", "porte_populacional",
           "perfil_ocupacao", "setor_dominante", "n_provedores_qtd",
           "provedor_lider", "concentracao", "faixa_modal"]))

## 5. Efeito da escolha do denominador

Comparação direta entre as duas penetrações. A diferença mede o peso dos imóveis não
ocupados que têm contrato ativo — casas de praia e de serra, tipicamente.

Se a troca de denominador reduzir drasticamente o número de municípios acima de 100%,
a hipótese está confirmada e o que parecia anomalia vira escolha metodológica.

In [0]:
acima_ocupados    = gold_mercado.filter(F.col("penetracao_ocupados_pct") > 100)
acima_enderecavel = gold_mercado.filter(F.col("penetracao_enderecavel_pct") > 100)

print(f"Municipios acima de 100% sobre domicilios OCUPADOS    : {acima_ocupados.count()} de {gold_mercado.count()}")
print(f"Municipios acima de 100% sobre domicilios RECENSEADOS : {acima_enderecavel.count()} de {gold_mercado.count()}")

display(exibir(
    gold_mercado.orderBy(F.col("penetracao_ocupados_pct").desc()).limit(25),
    numericos={"domicilios_total_qtd":       (0, ""),
               "domicilios_ocupados_qtd":    (0, ""),
               "domicilios_uso_ocasional_qtd": (0, ""),
               "uso_ocasional_pct":          (2, " %"),
               "acessos_total_qtd":          (0, ""),
               "penetracao_ocupados_pct":    (2, " %"),
               "penetracao_enderecavel_pct": (2, " %")},
    texto=["nome_municipio", "mesorregiao_nome", "perfil_ocupacao"]))

In [0]:
print("=== Penetracao media por perfil de ocupacao ===")
display(exibir(
    gold_mercado.groupBy("perfil_ocupacao").agg(
        F.count("*").alias("municipios_qtd"),
        F.round(F.avg("penetracao_ocupados_pct"), 2).alias("penetracao_ocupados_media_pct"),
        F.round(F.avg("penetracao_enderecavel_pct"), 2).alias("penetracao_enderecavel_media_pct"),
        F.round(F.avg("uso_ocasional_pct"), 2).alias("uso_ocasional_medio_pct"),
    ).orderBy(F.col("municipios_qtd").desc()),
    numericos={"municipios_qtd": (0, ""),
               "penetracao_ocupados_media_pct": (2, " %"),
               "penetracao_enderecavel_media_pct": (2, " %"),
               "uso_ocasional_medio_pct": (2, " %")},
    texto=["perfil_ocupacao"]))

Os municípios que permanecerem acima de 100% mesmo sobre o total recenseado exigem
outra explicação: múltiplos contratos por imóvel, ou acessos declarados no município
da prestadora em vez do assinante. Nenhuma das duas é verificável com dado público.

In [0]:
if acima_enderecavel.count() > 0:
    display(exibir(
        acima_enderecavel.orderBy(F.col("penetracao_enderecavel_pct").desc()),
        numericos={"domicilios_total_qtd":       (0, ""),
                   "acessos_total_qtd":          (0, ""),
                   "acessos_pf_qtd":             (0, ""),
                   "acessos_pj_qtd":             (0, ""),
                   "uso_ocasional_pct":          (2, " %"),
                   "penetracao_enderecavel_pct": (2, " %")},
        texto=["nome_municipio", "mesorregiao_nome", "perfil_ocupacao",
               "n_provedores_qtd"]))
else:
    print("Nenhum municipio acima de 100% sobre o denominador enderecavel.")

## 6. Tabela analítica por prestadora

Uma linha por CNPJ, com pegada geográfica e posição competitiva.

In [0]:
total_estadual = f.agg(F.sum("acessos_qtd")).first()[0]

lideranca = (
    share_empresa.filter(F.col("posicao") == 1)
    .groupBy("sk_empresa").agg(F.count("*").alias("municipios_lider_qtd"))
)

perfil_segmento = (
    base.groupBy("sk_empresa").agg(
        soma_se("e_pessoa_fisica",   "acessos_pf_qtd"),
        soma_se("e_pessoa_juridica", "acessos_pj_qtd"),
        soma_se("e_fibra",           "acessos_fibra_qtd"),
    )
)

gold_prestadora = (
    f.groupBy("sk_empresa").agg(
        F.sum("acessos_qtd").alias("acessos_total_qtd"),
        F.countDistinct("sk_municipio").alias("municipios_qtd"),
    )
    .join(dim_empresa, "sk_empresa")
    .join(lideranca, "sk_empresa", "left")
    .join(perfil_segmento, "sk_empresa", "left")
    .fillna(0, subset=["municipios_lider_qtd"])
    .withColumn("share_estadual_pct",
        F.round(100 * F.col("acessos_total_qtd") / F.lit(total_estadual), 3))
    .withColumn("acessos_por_municipio_qtd",
        F.round(F.col("acessos_total_qtd") / F.col("municipios_qtd"), 1))
    .withColumn("fibra_pct",
        F.round(100 * F.col("acessos_fibra_qtd") / F.col("acessos_total_qtd"), 2))
    .withColumn("pj_pct",
        F.round(100 * F.col("acessos_pj_qtd") / F.col("acessos_total_qtd"), 2))
    .withColumn("perfil_atuacao",
        F.when(F.col("municipios_qtd") == 1,  "Municipal")
         .when(F.col("municipios_qtd") <= 10, "Regional")
         .when(F.col("municipios_qtd") <= 50, "Multirregional")
         .otherwise("Estadual"))
    .select("sk_empresa", "cnpj", "empresa", "grupo_economico", "porte",
            "acessos_total_qtd", "acessos_pf_qtd", "acessos_pj_qtd",
            "share_estadual_pct", "municipios_qtd", "acessos_por_municipio_qtd",
            "municipios_lider_qtd", "fibra_pct", "pj_pct", "perfil_atuacao")
)

gravar_gold(gold_prestadora, "prestadora_atuacao",
    "Tabela analitica com uma linha por prestadora atuante no RS. Traz pegada "
    "geografica, share estadual, numero de municipios em que e lider, perfil "
    "tecnologico e composicao entre pessoa fisica e juridica.")

documentar_colunas("prestadora_atuacao", {
    "sk_empresa": "Chave surrogate da prestadora. Linhagem: gold.dim_empresa.",
    "cnpj": "CNPJ da prestadora, 14 digitos. Chave natural.",
    "empresa": "Nome comercial da prestadora.",
    "grupo_economico": "Grupo economico declarado. OUTROS para provedores regionais.",
    "porte": "Porte segundo a Anatel. Dominio: Pequeno Porte, Grande Porte.",
    "acessos_total_qtd": "Total de acessos da prestadora no RS. Unidade: acessos.",
    "acessos_pf_qtd": "Acessos de pessoa fisica. Unidade: acessos.",
    "acessos_pj_qtd": "Acessos de pessoa juridica. Unidade: acessos.",
    "share_estadual_pct": "Participacao no total de acessos do estado. "
        "Unidade: percentual. Dominio: 0 a 100.",
    "municipios_qtd": "Municipios do RS em que a prestadora tem acesso declarado. "
        "Unidade: municipios. Dominio: 1 a 497.",
    "acessos_por_municipio_qtd": "Media de acessos por municipio atendido. "
        "Unidade: acessos por municipio. Indica densidade da operacao.",
    "municipios_lider_qtd": "Municipios em que a prestadora e a maior em acessos. "
        "Unidade: municipios.",
    "fibra_pct": "Participacao da fibra na base da prestadora. Unidade: percentual. "
        "Dominio: 0 a 100. Indica modernidade da rede.",
    "pj_pct": "Participacao de pessoa juridica na base. Unidade: percentual. "
        "Dominio: 0 a 100. Indica orientacao corporativa da operacao.",
    "perfil_atuacao": "Classificacao por abrangencia geografica. Dominio: Municipal "
        "(1 municipio), Regional (2 a 10), Multirregional (11 a 50), Estadual "
        "(mais de 50).",
})

In [0]:
display(exibir(
    gold_prestadora.orderBy(F.col("acessos_total_qtd").desc()).limit(25),
    numericos={"acessos_total_qtd": (0, ""),
               "acessos_pf_qtd": (0, ""),
               "acessos_pj_qtd": (0, ""),
               "share_estadual_pct": (3, " %"),
               "municipios_qtd": (0, ""),
               "acessos_por_municipio_qtd": (1, ""),
               "municipios_lider_qtd": (0, ""),
               "fibra_pct": (2, " %"),
               "pj_pct": (2, " %")},
    texto=["cnpj", "empresa", "grupo_economico", "porte", "perfil_atuacao"]))

## 7. Índice de oportunidade comercial

Combina três sinais em um índice comparável entre municípios:

- **Mercado não atendido** — `domicilios_sem_acesso_qtd`, em percentil
- **Base defasada** — `defasagem_pct`, em percentil
- **Poder aquisitivo** — `pib_per_capita_brl`, em percentil

O uso de percentis evita que uma variável de escala maior domine o índice. Os três
componentes têm peso igual.

O denominador agora é o total recenseado, então municípios de veraneio deixam de ser
penalizados indevidamente.


In [0]:
def percentil(df, coluna, nome):
    """Percentil (0 a 1) no universo de municipios.

    partitionBy(lit(1)) declara particao unica explicitamente. E o mesmo calculo de
    uma janela global, mas evita o aviso de performance do Spark - que aqui nao se
    aplica, ja que o universo tem 497 linhas.
    """
    janela = Window.partitionBy(F.lit(1)).orderBy(F.col(coluna))
    return df.withColumn(nome, F.round(F.percent_rank().over(janela), 4))

oportunidade = gold_mercado.filter(F.col("acessos_total_qtd").isNotNull())
oportunidade = percentil(oportunidade, "domicilios_sem_acesso_qtd", "p_nao_atendido")
oportunidade = percentil(oportunidade, "defasagem_pct",             "p_defasagem")
oportunidade = percentil(oportunidade, "pib_per_capita_brl",        "p_renda")

oportunidade = (
    oportunidade
    .withColumn("indice_oportunidade",
        F.round(100 * (F.col("p_nao_atendido")
                       + F.col("p_defasagem")
                       + F.col("p_renda")) / 3, 1))
    .select("codigo_ibge", "nome_municipio", "mesorregiao_nome", "porte_populacional",
            "perfil_ocupacao",
            "domicilios_total_qtd", "domicilios_sem_acesso_qtd",
            "acessos_total_qtd", "penetracao_enderecavel_pct",
            "defasagem_pct", "fibra_pct", "pib_per_capita_brl",
            "n_provedores_qtd", "hhi", "concentracao",
            "p_nao_atendido", "p_defasagem", "p_renda", "indice_oportunidade")
)

gravar_gold(oportunidade, "indice_oportunidade",
    "Indice composto de oportunidade comercial por municipio. Combina tres "
    "componentes em percentil, com peso igual: mercado nao atendido "
    "(domicilios_sem_acesso_qtd sobre o total recenseado, com piso em zero), base "
    "tecnologica defasada (defasagem_pct) e poder aquisitivo (pib_per_capita_brl). "
    "Escala 0 a 100. Pesos iguais por escolha neutra declarada, sem base empirica "
    "para pondera-los de outra forma neste MVP. Ferramenta de priorizacao analitica, "
    "nao recomendacao de investimento.")

documentar_colunas("indice_oportunidade", {
    "codigo_ibge": "Codigo IBGE do municipio, 7 digitos. Chave natural.",
    "nome_municipio": "Nome oficial do municipio.",
    "mesorregiao_nome": "Mesorregiao geografica do IBGE.",
    "porte_populacional": "Faixa de porte populacional.",
    "perfil_ocupacao": "Classificacao pelo peso do uso ocasional. Dominio: Ocupacao "
        "permanente, Ocupacao mista, Veraneio ou turismo.",
    "domicilios_total_qtd": "Total de domicilios recenseados. Unidade: domicilios. "
        "Denominador de mercado enderecavel.",
    "domicilios_sem_acesso_qtd": "Mercado nao atendido, piso em zero. Unidade: "
        "domicilios. Componente 1 do indice.",
    "acessos_total_qtd": "Total de acessos no municipio. Unidade: acessos.",
    "penetracao_enderecavel_pct": "Acessos totais sobre domicilios recenseados. "
        "Unidade: percentual.",
    "defasagem_pct": "Participacao de acessos abaixo de 34 Mbps. Unidade: percentual. "
        "Componente 2 do indice.",
    "fibra_pct": "Participacao da fibra optica. Unidade: percentual.",
    "pib_per_capita_brl": "PIB por habitante. Unidade: reais. Componente 3 do indice.",
    "n_provedores_qtd": "CNPJs distintos atuando no municipio. Unidade: prestadoras.",
    "hhi": "Indice Herfindahl-Hirschman. Dominio: 0 a 10000.",
    "concentracao": "Classificacao do HHI. Dominio: Baixa, Moderada, Alta.",
    "p_nao_atendido": "Percentil de domicilios_sem_acesso_qtd. Dominio: 0 a 1.",
    "p_defasagem": "Percentil de defasagem_pct. Dominio: 0 a 1.",
    "p_renda": "Percentil de pib_per_capita_brl. Dominio: 0 a 1.",
    "indice_oportunidade": "Media simples dos tres percentis, multiplicada por 100. "
        "Unidade: indice adimensional. Dominio: 0 a 100.",
})

In [0]:
display(exibir(
    oportunidade.orderBy(F.col("indice_oportunidade").desc()).limit(30),
    numericos={"domicilios_total_qtd": (0, ""),
               "domicilios_sem_acesso_qtd": (0, ""),
               "acessos_total_qtd": (0, ""),
               "penetracao_enderecavel_pct": (2, " %"),
               "defasagem_pct": (2, " %"),
               "fibra_pct": (2, " %"),
               "pib_per_capita_brl": (2, ""),
               "hhi": (1, ""),
               "indice_oportunidade": (1, "")},
    texto=["codigo_ibge", "nome_municipio", "mesorregiao_nome",
           "porte_populacional", "perfil_ocupacao", "n_provedores_qtd",
           "concentracao"]))

## 8. Inventário da camada

In [0]:
print(f"{'TABELA':<24} {'LINHAS':>10} {'COLUNAS':>9}")
print("-" * 45)
for t in sorted(spark.sql(f"SHOW TABLES IN {CATALOGO_GOLD}.{SCHEMA}").collect(),
                key=lambda x: x.tableName):
    d = spark.table(f"{CATALOGO_GOLD}.{SCHEMA}.{t.tableName}")
    print(f"{t.tableName:<24} {d.count():>10,} {len(d.columns):>9}")